# CS 3110/5110: Data Privacy
## In-Class Exercise, 10/20/2025

In [1]:
# Load the data and libraries
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

def laplace_mech(v, sensitivity, epsilon):
    return v + np.random.laplace(loc=0, scale=sensitivity / epsilon)

def gaussian_mech(v, sensitivity, epsilon, delta):
    return v + np.random.normal(loc=0, scale=sensitivity * np.sqrt(2*np.log(1.25/delta)) / epsilon)

def pct_error(orig, priv):
    return np.abs(orig - priv)/orig * 100.0

adult = pd.read_csv('https://github.com/jnear/cs3110-data-privacy/raw/main/homework/adult_with_pii.csv')

## Question 1

The code below defines a finite set of options for marital status. Define a *scoring function* that returns high scores for common marital statuses, and low scores for uncommon ones (e.g. the score could be the number of people with that status).

In [2]:
options = ['Never-married', 'Married-civ-spouse', 'Divorced',
           'Married-spouse-absent', 'Separated', 'Married-AF-spouse',
           'Widowed']

def score(option):
    return len(adult[adult['Marital Status'] == option])

score('Never-married')

10683

In [3]:
# TEST CASE
assert score('Never-married') == 10683

## Question 2

Implement `report_noisy_max` using the Laplace mechanism. `report_noisy_max` should return the value in a set that approximately maximizes the value of the score function. It should *not* return the score itself.

In [4]:
def report_noisy_max(R, score, sensitivity, epsilon):
    # 1) assign score to each option
    scores = [score(r) for r in R]
    # 2) add noise to the score 
    noisy_scores = [laplace_mech(s, sensitivity=sensitivity, epsilon=epsilon) for s in scores]
    # 3) pick the option that has the highest noisy score 
    max_idz = np.argmax(noisy_scores)
    return R[max_idz]

report_noisy_max(options, score, 1, 1)

# When does this works well?
# - score function has low of sensitivity and lots signal
#       - lots of signal -> scores of different options are far appart
# - when you have lots of data ( = score function has a lot orignal)
# - many good scoring functions tend to be a counting query
#when does it not?
# - when the scoring function doesnt match the question you are asking
# - when the scoring function has not much signal compared to the noise
#  -  that happens when scoring function has not much data

'Married-civ-spouse'

In [5]:
# TEST CASE
assert report_noisy_max(options, score, 1, 1) == 'Married-civ-spouse'

## Question 3

What is the **total privacy cost** of `report_noisy_max` under *sequential composition*?

1. Sensitivity:
    - I add nosie to the result of the score function
    - We assume that the score fucntion has sensitivity of 'sensitivity'
2. Composition:
    - I run the laplae mechanism 'len(R)' times, each with a privacy cost of 'epsilon'
    - The function has a total privacy cost of 'epsilon * len(R)', by sequential composition
3. Post processing:
    - I take max oif the noisy scores ( they are all noisy)
    - I return the option at the t index ( the options are public, the index satisfies DP)


**Twist**: Exponential mechanism theorem says:

- The total privacy cost of report_noisy_max is just epsilon

Important: the privacy cost of 'report_noisy_max' does not change, no matter how many options you have

Why?
- The function returns less information that it *could*
- It returns only the identity of the options with the max score 

## Above Threshold

The following code implements Above Threshold:

In [6]:
# preserves epsilon-differential privacy
def above_threshold(queries, df, T, epsilon):
    T_hat = T + np.random.laplace(loc=0, scale = 2/epsilon)
    
    for idx, q in enumerate(queries):
        nu_i = np.random.laplace(loc=0, scale = 4/epsilon)
        if q(df) + nu_i >= T_hat:
            return idx
    return -1 # the index of the last element

## Question 4

Use `above_threshold` to find the first `age` for which `len(adult[adult['Age'] == age]) >= 800` (the first age for which more than 800 people have that age).

In [ ]:
def make_query(age):
    def q(df):
        return len(df[df['Age'] == age])
    return q

def find_first_age_above_800(epsilon):
        #1. construct the sequence of queries that alow me to asnwer question I care about
        qs = [make_query(age) for age in range(100)]
        #2. call above trashehold on the sequence of quiries and appropriate trashehold
        idx = above_threshold(qs, adult, T=800, epsilon=epsilon)
        #3. post process the index to get the asnwer i actualy want
        return idx #because age is equal to index in qs (not always the case)
    
find_first_age_above_800(0.1)

19

In [8]:
# TEST CASE
assert find_first_age_above_800(1.0) == 22

AssertionError: 

## Question 5

Use `above_threshold` to implement `pick_b` for the following summation query. `pick_b` should pick a clipping parameter `b`.

In [9]:
def make_query(b):
    def q(df):
        r1 = df['Age'].clip(lower=0, upper=b).sum()
        r2 = df['Age'].clip(lower=0, upper=b+1).sum()
        return r1 - r2
    return q

def pick_b(epsilon):
    #1. construct the sequence of queries that allow me to answer question i care about
    qs = [make_query(age) for age in range(1000000)]
    #2. call above threshold on the sequence of queries and appropriate threshold
    idx = above_threshold(qs, adult, T=0, epsilon=epsilon)
    #3. post process the index ti get the asnwer i actualy wat
    return idx #because age is equal to index in qs (not always the case)
    
pick_b(1.0)

90

In [10]:
# TEST CASE
b = pick_b(1.0)
assert b > 80
assert b < 100

## Question 6

Implement `above_threshold_val`, which returns the *value* of the first query result above the threshold. Your solution should have a **total privacy cost** of `epsilon`.

In [11]:
def make_query(age):
    def q(df):
        return len(df[df['Age'] == age])
    return q

def above_threshold_val(queries, df, T, epsilon):
    idx = above_threshold(queries, df, T=T, epsilon=epsilon/2)
    val = laplace_mech(queries[idx](df), sensitivity=1, epsilon=epsilon/2)
    return val

queries = [make_query(age) for age in range(0,100)]
above_threshold_val(queries, adult, 800, 1.0)

877.1913428179394

In [12]:
# TEST CASE
queries = [make_query(age) for age in range(0,100)]

results = [above_threshold_val(queries, adult, 800, 1.0) for _ in range(20)]
assert np.mean(results) > 865
assert np.mean(results) < 890

## Question 7

Argue that your solution in question 10 has a total privacy cost of `epsilon`.

YOUR ANSWER HERE